# FINS exp48 — learned arm for heat (Colab, GPU)

Retrains heat's CNO on the corrupted training data and runs FINS through it in exp48's configuration
(`exp48_learned.py`, the same driver as the sandbox). Conditions in order: clean-rt, n1e-4, t4, t8, t2, n3e-4,
n1e-3, n1e-2; seeds 42-44: 24 trainings. KdV and heat's clean condition are already done in the sandbox.

Runtime: T4 GPU. Checkpoints `S_theta_heat_s{s}_g64x128_cno_smooth_L1_{cond}.pt` and the outputs go to
`FINS_RESULTS_DIR` on Drive; training saves a partial state every 10 epochs, so after a disconnect rerun cells 1, 2
and 3: the loop skips finished runs and resumes the interrupted training exactly where it stopped.

Cells: 1 setup (Drive, repository, dependencies); 2 the four updated files (works with the repository zip already on
Drive); 3 the run; 4 progress; 5 zip of the outputs (npz and logs, no checkpoints) for download.

In [ ]:
# 1. setup: Drive, repository from the newest fins_repo_full*.zip on Drive that contains exp48, dependencies
import os, sys, subprocess, zipfile, glob, time, shutil
from google.colab import drive
drive.mount('/content/drive')
os.environ['FINS_RESULTS_DIR'] = '/content/drive/MyDrive/fins_runs/results'
os.makedirs(os.environ['FINS_RESULTS_DIR'], exist_ok=True)
# Drive keeps both files when a zip of the same name is uploaded again ('fins_repo_full (1).zip'): list them all
cands = sorted(glob.glob('/content/drive/MyDrive/fins_repo_full*.zip') + glob.glob('/content/drive/MyDrive/*/fins_repo_full*.zip'),
               key=os.path.getmtime, reverse=True)
ZIP_PATH = None
for z in cands:
    names = zipfile.ZipFile(z).namelist()
    has = 'fins_repo/experiments/corrupt.py' in names
    print(f'{z}  {time.ctime(os.path.getmtime(z))}  {os.path.getsize(z)} bytes  corrupt.py: {has}')
    if has and ZIP_PATH is None:
        ZIP_PATH = z
if ZIP_PATH is None:
    raise SystemExit('no fins_repo_full*.zip on Drive contains experiments/corrupt.py: upload the zip from the chat '
                     'to My Drive and rerun this cell')
print('using', ZIP_PATH)
shutil.rmtree('/content/fins', ignore_errors=True)             # never reuse a tree unpacked from an older zip
with zipfile.ZipFile(ZIP_PATH) as z:
    z.extractall('/content/fins')
REPO = '/content/fins/fins_repo'
EXPDIR = os.path.join(REPO, 'experiments')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'neuraloperator'], check=False)
import torch
print('torch', torch.__version__, '| cuda:', torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
print('repo:', REPO, '| results:', os.environ['FINS_RESULTS_DIR'])

### 2. Updated files
Resumable training (`save_partial` / `load_partial`), the heat CNO's partial checkpoint in exp40, and the driver
`exp48_learned.py`. Run all four cells.

In [ ]:
%%writefile /content/fins/fins_repo/pde/stepper_surrogate.py
"""stepper_surrogate -- time-stepping solution operator for FINS.

u(t) -> u(t + dt) as u + F(u), F a 1-D FNO without a coordinate channel,
so every step is translation-equivariant by construction; the trajectory
is the rollout, and jvp differentiates the product of step Jacobians
exactly. exp28 (small Burgers): 5-7x lower trajectory error than the
one-shot u0 -> u(., t) FNO and a more faithful Jacobian on the non-symmetry
u d_u, but NOT on Galilean (residual ~1.6 at 16, 128 and 512 training ICs).
Since the probe does not predict discovery (tanv3), the decisive test is a
FINS run through it. Fixed-grid equations only (Burgers, KdV): the step is
learned at the stored spacing, which varies per sample on multi-scale heat.
"""

import os

import torch
import torch.nn as nn
from neuralop.models import FNO


class Stepper(nn.Module):
    def __init__(self, modes=24, width=48, layers=4, positional_embedding="grid"):
        super().__init__()
        # neuralop's default "grid" embedding feeds x in [0, 1] as an input channel, which breaks
        # shift equivariance on the torus; None keeps the operator translation-equivariant
        self.f = FNO(n_modes=(modes,), hidden_channels=width, in_channels=1,
                     out_channels=1, n_layers=layers, positional_embedding=positional_embedding)

    def forward(self, u):
        return u + self.f(u)


class RolloutOperator(nn.Module):
    """(u0, x_coords, t_coords) -> [B, 1, Nx, Nt], the signature FINS uses."""

    def __init__(self, stepper, nt):
        super().__init__()
        self.step, self.nt = stepper, nt

    def forward(self, u0, x_coords=None, t_coords=None):
        u = u0 if u0.dim() == 3 else u0.unsqueeze(1)
        out = [u]
        for _ in range(self.nt - 1):
            u = self.step(u)
            out.append(u)
        return torch.stack(out, dim=-1)


def _increment_terms(stp, sols, idx, j0, unroll):
    """Unrolled errors and increments: per sample and step k, ||pred_k -
    true_k||^2 and ||true_k - true_0||^2."""
    u0 = sols[idx, :, :, j0]
    u = u0
    errs, incs = [], []
    for k in range(1, unroll + 1):
        u = stp(u)
        tk = sols[idx, :, :, j0 + k]
        errs.append((u - tk).pow(2).mean(dim=(1, 2)))
        incs.append((tk - u0).pow(2).mean(dim=(1, 2)))
    return torch.stack(errs, 1), torch.stack(incs, 1)


def _relative(errs, incs, floor=0.1):
    """Error relative to the increment, with a floor at a fraction of the
    batch-mean increment so near-stationary windows cannot dominate."""
    den = incs + floor * incs.mean(dim=0, keepdim=True) + 1e-12
    return errs / den


def save_partial(path, ep, model, opt, sched, best, best_state):
    """Training state after epoch ep (atomic write): resuming reproduces an uninterrupted run."""
    state = dict(epoch=ep, model={k: t for k, t in model.state_dict().items() if torch.is_tensor(t)},
                 opt=opt.state_dict(), sched=sched.state_dict(), best=best, best_state=best_state,
                 rng=torch.get_rng_state(),
                 cuda_rng=torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None)
    torch.save(state, path + ".tmp")
    os.replace(path + ".tmp", path)


def load_partial(path, model, opt, sched):
    """Restore a save_partial state; returns (first epoch to run, best, best_state)."""
    state = torch.load(path, map_location="cpu", weights_only=False)
    model.load_state_dict(state["model"])
    opt.load_state_dict(state["opt"])
    sched.load_state_dict(state["sched"])
    torch.set_rng_state(state["rng"])
    if state["cuda_rng"] is not None and torch.cuda.is_available():
        torch.cuda.set_rng_state_all(state["cuda_rng"])
    return state["epoch"] + 1, state["best"], state["best_state"]


def train_stepper(train_sol, device, epochs=400, unroll=4, lr=2e-3,
                  batch=64, seed=0, verbose=True, loss="abs", val_frac=0.1,
                  eval_every=10, ckpt=None, ckpt_every=10, **arch):
    """Multi-step (unrolled) training on windows of the stored trajectories.

    loss="abs": MSE on u (the original objective). loss="incr": error of the
    k-step increment relative to the increment itself. The symmetry criterion
    compares increments, and on slow dynamics an accurate trajectory can
    still carry large increment error (heat: 0.16 percent of u but ~40-70
    percent of the one-step increment under loss="abs").
    A fixed held-out set of windows is scored with the relative increment
    error every `eval_every` epochs and the best state is returned; the
    per-epoch training loss uses random windows and is too noisy for model
    selection. ckpt: path of a partial state written every ckpt_every
    epochs and resumed from if present (same result as an uninterrupted run).
    """
    torch.manual_seed(seed)
    sols = train_sol.to(device)
    n_all, _, nx, nt = sols.shape
    n_val = max(1, int(round(val_frac * n_all))) if val_frac > 0 else 0
    tr, va = sols[:n_all - n_val], sols[n_all - n_val:]
    n = tr.shape[0]
    stp = Stepper(**arch).to(device)
    opt = torch.optim.Adam(stp.parameters(), lr=lr)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, epochs)
    starts = torch.arange(0, nt - unroll, max((nt - unroll) // 16, 1), device=device)
    best, best_state = float("inf"), None

    def validate():
        if n_val == 0:
            return float("nan")
        stp.eval()
        with torch.no_grad():
            vals = []
            for j in starts.tolist():
                idx = torch.arange(n_val, device=device)
                e, c = _increment_terms(stp, va, idx, torch.full_like(idx, j), unroll)
                vals.append(_relative(e, c))
            v = torch.cat(vals).mean().sqrt().item()
        stp.train()
        return v

    first = 0
    if ckpt and os.path.exists(ckpt):
        first, best, best_state = load_partial(ckpt, stp, opt, sched)
        if verbose:
            print(f"      stepper: resumed from {ckpt} after epoch {first}", flush=True)
    for ep in range(first, epochs):
        perm = torch.randperm(n, device=device)
        tot = 0.0
        for i in range(0, n, batch):
            idx = perm[i:i + batch]
            j0 = torch.randint(0, nt - unroll, (idx.numel(),), device=device)
            e, c = _increment_terms(stp, tr, idx, j0, unroll)
            l = _relative(e, c).mean() if loss == "incr" else e.mean()
            opt.zero_grad()
            l.backward()
            opt.step()
            tot += l.item() * idx.numel()
        sched.step()
        last = ep + 1 == epochs
        if (ep + 1) % eval_every == 0 or last:
            v = validate()
            if v < best:
                best = v
                best_state = {k: t.detach().clone() for k, t in stp.state_dict().items()
                              if torch.is_tensor(t)}
        if verbose and (ep == 0 or (ep + 1) % max(epochs // 10, 1) == 0):
            v = validate()
            print(f"      stepper epoch {ep + 1}/{epochs}: train {loss} loss "
                  f"{tot / n:.3e} | held-out increment error {v:.3f} "
                  f"(best {best:.3f})", flush=True)
        if ckpt and (ep + 1) % ckpt_every == 0 and not last:
            save_partial(ckpt, ep, stp, opt, sched, best, best_state)
    if best_state is not None:
        # tensors only: neuralop's state_dict carries a '_metadata' entry that
        # its load_state_dict rejects
        stp.load_state_dict(best_state)
        if verbose:
            print(f"      stepper: returning the best held-out state "
                  f"(increment error {best:.3f})", flush=True)
    return stp

In [ ]:
%%writefile /content/fins/fins_repo/pde/window_stepper.py
"""Window stepper: a local operator that needs no boundary condition.

Valid (unpadded) 1-D convolutions map u(t) on a window of width W to
u(t + dt) on the central W - 2r points, r the receptive radius. After K
steps the output lives on W - 2rK points: the influence cone is built into
the architecture, and the operator never closes the domain. Trained on
windows cut from trajectories; periodic data may be read as solutions on
the line, windows spanning several periods.

GlobalStepper: same-size, non-periodic global operators on the same windows
(CNO; FNO with zero domain padding and no positional embedding). r = 0: the
output keeps the window's width, and only an interior region is scored.
"""

import os

import torch
import torch.nn as nn
import torch.nn.functional as F
from neuralop.models import FNO

from cno1d import CNO1d
from stepper_surrogate import load_partial, save_partial


class WindowStepper(nn.Module):
    def __init__(self, width=64, layers=3, kernel=5):
        super().__init__()
        self.k = kernel // 2
        self.r = layers * self.k
        self.lift = nn.Conv1d(1, width, 1)
        self.body = nn.ModuleList([nn.Conv1d(width, width, kernel) for _ in range(layers)])
        self.proj = nn.Sequential(nn.Conv1d(width, width, 1), nn.GELU(), nn.Conv1d(width, 1, 1))

    def forward(self, u):
        h = self.lift(u)
        for conv in self.body:
            h = F.gelu(conv(h)) + h[:, :, self.k:-self.k]
        return u[:, :, self.r:-self.r] + self.proj(h)


def windows(sols, idx, j0, pos, width, steps):
    """Windows [B, width] at time j0 and targets for 1..steps, wrapping periodic data."""
    Nx = sols.shape[2]
    cols = (pos[:, None] + torch.arange(width, device=sols.device)[None]) % Nx
    rows = idx[:, None].expand_as(cols)
    return [sols[rows, 0, cols, (j0 + k)[:, None].expand_as(cols)] for k in range(steps + 1)]


def train_window_stepper(train_sol, device, epochs=400, unroll=4, out=32, lr=2e-3, batch=256,
                         seed=0, val_frac=0.1, eval_every=10, verbose=True, **arch):
    """Unrolled MSE on windows; returns the state with the best held-out increment error."""
    torch.manual_seed(seed)
    sols = train_sol.to(device)
    n_all, _, Nx, Nt = sols.shape
    n_val = max(1, int(round(val_frac * n_all)))
    stp = WindowStepper(**arch).to(device)
    width = out + 2 * stp.r * unroll
    opt = torch.optim.Adam(stp.parameters(), lr=lr)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, epochs)
    n_tr = n_all - n_val
    best, best_state = float("inf"), None

    def rollout(idx, j0, pos):
        ws = windows(sols, idx, j0, pos, width, unroll)
        s = ws[0][:, None]
        errs, incs = [], []
        for k in range(1, unroll + 1):
            s = stp(s)
            c = k * stp.r
            tgt = ws[k][:, c:width - c]
            errs.append((s[:, 0] - tgt).pow(2).mean(1))
            incs.append((tgt - ws[0][:, c:width - c]).pow(2).mean(1))
        return torch.stack(errs, 1), torch.stack(incs, 1)

    g = torch.Generator(device="cpu").manual_seed(seed + 1)
    vidx = torch.arange(n_tr, n_all).repeat_interleave(16).to(device)
    vj = torch.randint(0, Nt - unroll, (vidx.numel(),), generator=g).to(device)
    vp = torch.randint(0, Nx, (vidx.numel(),), generator=g).to(device)

    def validate():
        stp.eval()
        with torch.no_grad():
            e, c = rollout(vidx, vj, vp)
            v = float((e / (c + 0.1 * c.mean(0, keepdim=True) + 1e-12)).mean().sqrt())
        stp.train()
        return v

    per_epoch = max(n_tr * 8 // batch, 1)
    for ep in range(epochs):
        tot = 0.0
        for _ in range(per_epoch):
            idx = torch.randint(0, n_tr, (batch,), device=device)
            j0 = torch.randint(0, Nt - unroll, (batch,), device=device)
            pos = torch.randint(0, Nx, (batch,), device=device)
            e, _ = rollout(idx, j0, pos)
            loss = e.mean()
            opt.zero_grad()
            loss.backward()
            opt.step()
            tot += loss.item()
        sched.step()
        if (ep + 1) % eval_every == 0 or ep + 1 == epochs:
            v = validate()
            if v < best:
                best = v
                best_state = {k: t.detach().clone() for k, t in stp.state_dict().items()}
        if verbose and (ep == 0 or (ep + 1) % max(epochs // 10, 1) == 0):
            print(f"      window stepper epoch {ep + 1}/{epochs}: train loss {tot / per_epoch:.3e} | "
                  f"held-out increment error {validate():.3f} (best {best:.3f})", flush=True)
    if best_state is not None:
        stp.load_state_dict(best_state)
        if verbose:
            print(f"      window stepper: returning the best held-out state (increment error {best:.3f})",
                  flush=True)
    return stp


class GlobalStepper(nn.Module):
    """u -> u + net(u) on a window of fixed width; net is 'cno' or 'fnopad'."""

    def __init__(self, kind="cno", size=256, cno_layers=3, cno_res=2, cno_mult=16,
                 fno_modes=32, fno_width=48, fno_layers=4, fno_pad=0.25):
        super().__init__()
        self.r = 0
        self.kind = kind
        if kind == "cno":
            self.net = CNO1d(1, 1, size, cno_layers, N_res=cno_res, N_res_neck=cno_res,
                             channel_multiplier=cno_mult, use_bn=False)
        elif kind == "fnopad":
            self.net = FNO(n_modes=(fno_modes,), hidden_channels=fno_width, in_channels=1, out_channels=1,
                           n_layers=fno_layers, domain_padding=fno_pad, positional_embedding=None)
        else:
            raise ValueError(kind)

    def forward(self, u):
        return u + self.net(u)


def to_float64(module):
    """Module.double() leaves complex parameters (spectral weights) in single precision."""
    module.double()
    for p in module.parameters():
        if p.is_complex():
            p.data = p.data.to(torch.complex128)
    for name, b in module.named_buffers():
        if b.is_complex():
            b.data = b.data.to(torch.complex128)
    return module


def train_global_stepper(train_sol, device, kind="cno", size=256, epochs=400, unroll=4, margin=6,
                         lr=1e-3, batch=256, seed=0, val_frac=0.1, eval_every=10, verbose=True, ckpt=None,
                         ckpt_every=10, **arch):
    """Unrolled MSE on windows of `size` points; step k is scored on [margin k, size - margin k],
    where information from outside the window cannot yet have arrived. Returns the state with
    the best held-out increment error on the same interior. ckpt: partial state written every
    ckpt_every epochs and resumed from if present (same result as an uninterrupted run)."""
    torch.manual_seed(seed)
    sols = train_sol.to(device)
    n_all, _, Nx, Nt = sols.shape
    n_val = max(1, int(round(val_frac * n_all)))
    n_tr = n_all - n_val
    stp = GlobalStepper(kind, size, **arch).to(device)
    opt = torch.optim.Adam(stp.parameters(), lr=lr)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, epochs)
    best, best_state = float("inf"), None

    def rollout(idx, j0, pos):
        ws = windows(sols, idx, j0, pos, size, unroll)
        s = ws[0][:, None]
        errs, incs = [], []
        for k in range(1, unroll + 1):
            s = stp(s)
            c = k * margin
            tgt = ws[k][:, c:size - c]
            errs.append((s[:, 0, c:size - c] - tgt).pow(2).mean(1))
            incs.append((tgt - ws[0][:, c:size - c]).pow(2).mean(1))
        return torch.stack(errs, 1), torch.stack(incs, 1)

    g = torch.Generator(device="cpu").manual_seed(seed + 1)
    vidx = torch.arange(n_tr, n_all).repeat_interleave(16).to(device)
    vj = torch.randint(0, Nt - unroll, (vidx.numel(),), generator=g).to(device)
    vp = torch.randint(0, Nx, (vidx.numel(),), generator=g).to(device)

    def validate():
        stp.eval()
        with torch.no_grad():
            e, c = rollout(vidx, vj, vp)
            v = float((e / (c + 0.1 * c.mean(0, keepdim=True) + 1e-12)).mean().sqrt())
        stp.train()
        return v

    per_epoch = max(n_tr * 8 // batch, 1)
    first = 0
    if ckpt and os.path.exists(ckpt):
        first, best, best_state = load_partial(ckpt, stp, opt, sched)
        if verbose:
            print(f"      {kind} stepper: resumed from {ckpt} after epoch {first}", flush=True)
    for ep in range(first, epochs):
        tot = 0.0
        for _ in range(per_epoch):
            idx = torch.randint(0, n_tr, (batch,), device=device)
            j0 = torch.randint(0, Nt - unroll, (batch,), device=device)
            pos = torch.randint(0, Nx, (batch,), device=device)
            e, _ = rollout(idx, j0, pos)
            loss = e.mean()
            opt.zero_grad()
            loss.backward()
            opt.step()
            tot += loss.item()
        sched.step()
        if (ep + 1) % eval_every == 0 or ep + 1 == epochs:
            v = validate()
            if v < best:
                best = v
                best_state = {k: t.detach().clone() for k, t in stp.state_dict().items() if torch.is_tensor(t)}
        if verbose and (ep == 0 or (ep + 1) % max(epochs // 10, 1) == 0):
            print(f"      {kind} stepper epoch {ep + 1}/{epochs}: train loss {tot / per_epoch:.3e} | "
                  f"held-out increment error {validate():.3f} (best {best:.3f})", flush=True)
        if ckpt and (ep + 1) % ckpt_every == 0 and ep + 1 < epochs:
            save_partial(ckpt, ep, stp, opt, sched, best, best_state)
    if best_state is not None:
        stp.load_state_dict(best_state)
        if verbose:
            print(f"      {kind} stepper: returning the best held-out state (increment error {best:.3f})", flush=True)
    return stp

In [ ]:
%%writefile /content/fins/fins_repo/experiments/exp40_window.py
"""Experiment 40 -- discovery through a boundary-free window operator.

Periodic trajectories are read as solutions on the line: a window spans
several periods with a continuous coordinate, so no characteristic jumps
anywhere (projective's u-independent x / 2 pi included). u_x is a
fourth-order difference inside the window; the window stepper's K-step
output lives on the central W - 2rK points, where every step is scored.
Same mu-normalised eigenproblem as exp34; ground truth scores only.

--noise / --thin (exp48): every --thin-th snapshot and Gaussian noise of --noise times each trajectory's standard
deviation (corrupt.py), on the evaluation data and on a trained operator's training data; u_t by fourth-order
differences of the corrupted snapshots; K and --every count kept snapshots; the heat kernel is composed --thin times
(still exact). Saved u: the clean evaluation data.

Run:  python exp40_window.py --pde burgers --train-if-missing --epochs 400
"""

import argparse
import itertools
import os
import sys

# progress bars off unless TQDM_DISABLE is set explicitly (must precede tqdm's import)
os.environ.setdefault("TQDM_DISABLE", "1")

import numpy as np
import torch
from torch.func import jvp

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath(__file__)), "..", "pde"))
import exp31_polynomial_discovery as e31                   # noqa: E402
import exp32_blind_readout as e32                          # noqa: E402
import exp34_normalization as e34                          # noqa: E402
import corrupt as cr                                       # noqa: E402
from window_stepper import (GlobalStepper, WindowStepper, to_float64,  # noqa: E402
                            train_global_stepper, train_window_stepper)


def moment_kernel(sigma2, r):
    """Symmetric (2r+1)-tap kernel whose even moments 0..2r match the heat step
    exp(-sigma2 theta^2) (Gaussian of variance 2 sigma2): exact to O(theta^(2r+2))."""
    n = np.arange(r + 1)
    V = np.array([[(1.0 if nn == 0 else 0.0) if i == 0 else 2.0 * i ** (2 * nn) for i in range(r + 1)]
                  for nn in n])
    dfact = np.array([np.prod(np.arange(2 * nn - 1, 0, -2)) if nn else 1.0 for nn in n])
    w = np.linalg.solve(V, dfact * (2 * sigma2) ** n)
    return np.concatenate([w[:0:-1], w])


def truncated_kernel(sigma2, r, Nx):
    m = np.fft.fftfreq(Nx, d=1.0 / Nx)
    h = np.real(np.fft.ifft(np.exp(-sigma2 * (2 * np.pi * m / Nx) ** 2)))
    return np.concatenate([h[-r:], h[:r + 1]])


class HeatKernelWindow(torch.nn.Module):
    """Exact-by-construction heat step on the line as a valid convolution."""

    def __init__(self, ker):
        super().__init__()
        self.register_buffer("w", torch.tensor(np.asarray(ker)[::-1].copy())[None, None])
        self.r = (len(ker) - 1) // 2

    def forward(self, u):
        return torch.nn.functional.conv1d(u, self.w)


def trajectories(pde, cfg, n, seed, smooth=False, only_L=None):
    """xs [n, Nx], ts [n, Nt], u [n, Nx, Nt], u_t. Heat: smooth = benchmark initial data low-passed
    and evolved exactly; only_L = keep one domain length (the benchmark cycles five, so 5 n are drawn)."""
    if pde == "heat" and (smooth or only_L):
        n_gen = 5 * n if only_L else n
        ics, sols, xs_t, ts_t = e31.generate_pde_data("heat", cfg, n_gen, seed=seed)
        xs, ts = xs_t[:, 0].numpy().astype(np.float64), ts_t[:, 0].numpy().astype(np.float64)
        if smooth:
            # benchmark initial data low-passed (no energy near Nyquist), evolved exactly
            Nx = xs.shape[1]
            m = np.fft.fftfreq(Nx, d=1.0 / Nx)
            c0 = np.fft.fft(ics[:, 0].numpy().astype(np.float64), axis=-1) * np.exp(-(m / (Nx / 6)) ** 8)
            k2 = (2 * np.pi * m[None] / e31.lengths(xs)[:, None]) ** 2
            u = np.real(np.fft.ifft(c0[:, :, None] * np.exp(-cfg.alpha_heat * k2[:, :, None] * ts[:, None, :]),
                                    axis=1))
        else:
            u = sols[:, 0].numpy().astype(np.float64)
        if only_L:
            sel = np.flatnonzero(np.isclose(e31.lengths(xs), only_L))[:n]
            if len(sel) < n:
                raise SystemExit(f"only {len(sel)} heat samples with L = {only_L}")
            xs, ts, u = xs[sel], ts[sel], u[sel]
        return xs, ts, u, e31.time_derivative(u, ts, "central4")
    if pde == "kdv":
        return e31.kdv_stencil(cfg, n, seed, 2e-5)
    if pde == "burgers":
        xs, ts, u = e31.burgers_coarse(cfg, n, seed)
    else:
        _, sols, xs_t, ts_t = e31.generate_pde_data(pde, cfg, n, seed=seed)
        xs, ts = xs_t[:, 0].numpy().astype(np.float64), ts_t[:, 0].numpy().astype(np.float64)
        u = sols[:, 0].numpy().astype(np.float64)
    return xs, ts, u, e31.time_derivative(u, ts, "central4")


def dx4(u, dx):
    """Fourth-order central difference along axis 1; second order at the two edge points."""
    o = np.empty_like(u)
    o[:, 2:-2] = (-u[:, 4:] + 8 * u[:, 3:-1] - 8 * u[:, 1:-3] + u[:, :-4]) / 12
    o[:, 1] = (u[:, 2] - u[:, 0]) / 2
    o[:, -2] = (u[:, -1] - u[:, -3]) / 2
    o[:, 0] = (-3 * u[:, 0] + 4 * u[:, 1] - u[:, 2]) / 2
    o[:, -1] = (3 * u[:, -1] - 4 * u[:, -2] + u[:, -3]) / 2
    return o / dx[:, None, None]


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--pde", default="burgers", choices=["burgers", "heat", "kdv"])
    ap.add_argument("--op", default="window", choices=["window", "cno", "fnopad", "heat-kernel"],
                    help="window: local valid-convolution stepper; cno / fnopad: same-size non-periodic "
                         "global steppers; heat-kernel: the exact heat map as a window operator")
    ap.add_argument("--seed", type=int, default=42)
    ap.add_argument("--n", type=int, default=6)
    ap.add_argument("--periods", type=int, default=4)
    ap.add_argument("--smooth", action="store_true",
                    help="heat: low-passed initial data, evolved exactly (evaluation, and training when a "
                         "same-size stepper is trained here)")
    ap.add_argument("--kernel-fit", default="moments", choices=["moments", "truncated"])
    ap.add_argument("--ux", default="fd", choices=["fd", "spectral"],
                    help="spectral: exact derivative of the periodic data (isolates the operator)")
    ap.add_argument("--degree", type=int, default=None)
    ap.add_argument("--K", type=int, default=16)
    ap.add_argument("--every", type=int, default=16)
    ap.add_argument("--mu", type=float, default=1.0)
    ap.add_argument("--m", type=int, default=0, help="blind readout of the m lowest (0: off)")
    ap.add_argument("--stepper", default=None)
    ap.add_argument("--tag", default="")
    ap.add_argument("--train-if-missing", action="store_true")
    ap.add_argument("--n-train", type=int, default=None)
    ap.add_argument("--epochs", type=int, default=400)
    ap.add_argument("--width", type=int, default=64)
    ap.add_argument("--layers", type=int, default=3)
    ap.add_argument("--kernel", type=int, default=5)
    ap.add_argument("--unroll", type=int, default=4, help="training rollout length")
    ap.add_argument("--score-margin", type=int, default=None,
                    help="same-size operators: points excluded at each window edge (default 6 K, the "
                         "window stepper's r = 6 cone, so all operators are scored on the same region)")
    ap.add_argument("--train-margin", type=int, default=6, help="same-size operators: training crop per step")
    ap.add_argument("--cno-layers", type=int, default=3)
    ap.add_argument("--cno-res", type=int, default=2)
    ap.add_argument("--cno-mult", type=int, default=16)
    ap.add_argument("--fno-modes", type=int, default=32)
    ap.add_argument("--fno-width", type=int, default=48)
    ap.add_argument("--fno-layers", type=int, default=4)
    ap.add_argument("--fno-pad", type=float, default=0.25)
    ap.add_argument("--lr", type=float, default=None, help="training learning rate (default per operator)")
    ap.add_argument("--data-seed", type=int, default=None,
                    help="seed of the evaluation trajectories (default --seed: inside the training set)")
    ap.add_argument("--only-L", type=float, default=None,
                    help="heat: keep one domain length (training and evaluation data)")
    ap.add_argument("--save", default=None, help="npz with the eigen-generators and the evaluation data")
    ap.add_argument("--reliability-filter", type=float, default=None,
                    help="drop trajectories whose one-step error exceeds this multiple of the median")
    ap.add_argument("--noise", type=float, default=0.0, help="Gaussian noise, times each trajectory's std (corrupt.py)")
    ap.add_argument("--thin", type=int, default=1, help="keep every thin-th snapshot (K, --every count kept ones)")
    ap.add_argument("--interior", action="store_true",
                    help="windows start at snapshot 2 and end by Nt - 3 (no one-sided u_t; exp48)")
    args = ap.parse_args()
    cfg = e31.ExperimentConfig()
    d = args.degree or (3 if args.pde == "heat" else 2)
    arch = dict(width=args.width, layers=args.layers, kernel=args.kernel)
    dev = "cuda" if torch.cuda.is_available() else "cpu"

    if args.op == "heat-kernel":
        xs0, ts0, _, _ = trajectories("heat", cfg, 1, args.seed)
        dx0, dt0 = e31.lengths(xs0)[0] / cfg.grid_size, ts0[0, 1] - ts0[0, 0]
        sig2 = cfg.alpha_heat * dt0 / dx0 ** 2
        ker = moment_kernel(sig2, 6) if args.kernel_fit == "moments" else truncated_kernel(sig2, 6, cfg.grid_size)
        th = np.linspace(0, np.pi, 2001)
        M = ker[6] + 2 * sum(ker[6 + i] * np.cos(i * th) for i in range(1, 7))
        band = th <= 2 * np.pi / 6
        tgt = np.exp(-sig2 * th ** 2)
        print(f"kernel ({args.kernel_fit}): sum {ker.sum():.12f}; multiplier error relative to the "
              f"increment over the data band {np.max(np.abs(M - tgt)[band][1:] / (1 - tgt[band][1:])):.1e}; "
              f"max |multiplier| on [0, pi] {np.abs(M).max():.3f}")
        base = ker
        for _ in range(args.thin - 1):           # one step between kept snapshots = thin original steps
            ker = np.convolve(ker, base)
        stp = HeatKernelWindow(ker)
    elif args.op in ("cno", "fnopad"):
        Wn = args.periods * cfg.grid_size
        garch = dict(cno_layers=args.cno_layers, cno_res=args.cno_res, cno_mult=args.cno_mult,
                     fno_modes=args.fno_modes, fno_width=args.fno_width, fno_layers=args.fno_layers,
                     fno_pad=args.fno_pad)
        name = args.stepper or (f"S_theta_{args.pde}_s{args.seed}_g{cfg.grid_size}x{cfg.time_steps}"
                                f"_{args.op}{args.tag}.pt")
        path = name if os.path.isabs(name) else os.path.join(os.environ.get("FINS_RESULTS_DIR", "."), name)
        stp = GlobalStepper(args.op, Wn, **garch)
        if os.path.exists(path):
            sd = torch.load(path, map_location="cpu", weights_only=False)
            stp.load_state_dict({k: v for k, v in sd.items() if torch.is_tensor(v)})
        elif args.train_if_missing:
            n_tr = args.n_train or cfg.n_train
            tr = trajectories(args.pde, cfg, n_tr, args.seed, args.smooth, args.only_L)[2]
            tr = torch.tensor(cr.corrupt(tr, args.noise, args.thin, args.seed, cr.TRAIN)[1], dtype=torch.float32)[:, None]
            print(f"training {args.pde} {args.op} stepper on {n_tr} trajectories" + (" (smooth)" if args.smooth else "")
                  + f", seed {args.seed}, "
                  f"{args.epochs} epochs, window {Wn}, unroll {args.unroll}, {dev}", flush=True)
            # partial state every 10 epochs next to the checkpoint: an interrupted run resumes exactly
            trained = train_global_stepper(tr, dev, kind=args.op, size=Wn, epochs=args.epochs,
                                           unroll=args.unroll, margin=args.train_margin, seed=args.seed,
                                           lr=args.lr or 1e-3, ckpt=path + ".partial", **garch)
            sd = {k: v.cpu() for k, v in trained.state_dict().items() if torch.is_tensor(v)}
            stp.load_state_dict(sd)
            torch.save(sd, path)
            if os.path.exists(path + ".partial"):
                os.remove(path + ".partial")
            print(f"saved {path}", flush=True)
        else:
            raise SystemExit(f"no {args.op} stepper at {path}; pass --train-if-missing")
    else:
        name = args.stepper or (f"S_theta_{args.pde}_s{args.seed}_g{cfg.grid_size}x{cfg.time_steps}"
                                f"_window{args.tag}.pt")
        path = name if os.path.isabs(name) else os.path.join(os.environ.get("FINS_RESULTS_DIR", "."), name)
        stp = WindowStepper(**arch)
        if os.path.exists(path):
            stp.load_state_dict(torch.load(path, map_location="cpu"))
        elif args.train_if_missing:
            n_tr = args.n_train or cfg.n_train
            tr = trajectories(args.pde, cfg, n_tr, args.seed)[2]
            tr = torch.tensor(cr.corrupt(tr, args.noise, args.thin, args.seed, cr.TRAIN)[1], dtype=torch.float32)[:, None]
            print(f"training {args.pde} window stepper on {n_tr} trajectories, seed {args.seed}, "
                  f"{args.epochs} epochs, radius {stp.r}, unroll {args.unroll}, {dev}", flush=True)
            trained = train_window_stepper(tr, dev, epochs=args.epochs, seed=args.seed,
                                           unroll=args.unroll, **arch)
            stp.load_state_dict({k: v.cpu() for k, v in trained.state_dict().items()})
            torch.save(stp.state_dict(), path)
            print(f"saved {path}", flush=True)
        else:
            raise SystemExit(f"no window stepper at {path}; pass --train-if-missing")
    stp = to_float64(stp).eval()
    shrink = stp.r * args.K if stp.r > 0 else (args.score_margin or 6 * args.K)
    for p in stp.parameters():
        p.requires_grad_(False)

    dseed = args.seed if args.data_seed is None else args.data_seed
    xs, ts, u, ut = trajectories(args.pde, cfg, args.n, dseed, args.smooth, args.only_L)
    u_clean, u = cr.corrupt(u, args.noise, args.thin, dseed, cr.EVAL)
    ts = ts[:, ::args.thin]
    if cr.corrupted(args):
        ut = e31.time_derivative(u, ts, "central4")
    tm = max(1, 8 // args.thin)                 # scored times: the same physical margin as 8 of the original
    su, sut = e31.band_shares(u, ut)
    print(f"  band check, energy above k = Nx/4: u {su:.1e}, u_t {sut:.1e}"
          + ("   FLAG: u_t lives where the operator has seen little" if sut > 1e-2 else ""))
    # equation-free reliability: one-step error on the scored region of each trajectory
    Nx0 = u.shape[1]
    W0 = args.periods * Nx0
    idx0 = np.arange(W0) % Nx0
    with torch.no_grad():
        errs = []
        for r in range(u.shape[0]):
            cur = torch.tensor(u[r][idx0].T[:-1])[:, None]
            pred = stp(cur)[:, 0].numpy()[:, shrink - stp.r:W0 - shrink - stp.r]
            tgt = u[r][idx0].T[1:, shrink:W0 - shrink]
            inc = tgt - u[r][idx0].T[:-1, shrink:W0 - shrink]
            errs.append(float(np.linalg.norm(pred - tgt) / np.linalg.norm(inc)))
    errs = np.array(errs)
    print("  per-trajectory one-step error: " + " ".join(f"{e:.3f}" for e in errs))
    if args.reliability_filter:
        keep_r = errs <= args.reliability_filter * np.median(errs)
        xs, ts, u, ut, u_clean = xs[keep_r], ts[keep_r], u[keep_r], ut[keep_r], u_clean[keep_r]
        print(f"  reliability filter: kept {int(keep_r.sum())} of {len(errs)} (<= {args.reliability_filter} x median)")
    N, Nx, Nt = u.shape
    W = args.periods * Nx
    if W - 2 * shrink < Nx // 2:
        raise SystemExit(f"window {W} too narrow for radius {stp.r} x K {args.K}")
    idx = np.arange(W) % Nx
    dx = e31.lengths(xs) / Nx
    xw = dx[:, None] * np.arange(W)[None]
    uw, utw = u[:, idx], ut[:, idx]
    if args.ux == "spectral":
        k = 2 * np.pi * np.fft.fftfreq(Nx, d=1.0)[None, :, None] / dx[:, None, None]
        uxw = np.real(np.fft.ifft(1j * k * np.fft.fft(u, axis=1), axis=1))[:, idx]
    else:
        uxw = dx4(uw, dx)

    Lw, T = (W * dx).max(), ts[:, -1].max()
    X = (xw[:, :, None] - xw[:, W // 2:W // 2 + 1, None]) / Lw
    Tn = ts[:, None, :] / T
    Un = uw / uw.std()
    mons = [mm for mm in itertools.product(range(d + 1), repeat=3) if sum(mm) <= d]
    F, Qb = [], []
    for comp in range(3):
        for a, b, c in mons:
            f = np.broadcast_to(X ** a * Tn ** b, uw.shape) * Un ** c
            fld = np.zeros((3,) + uw.shape)
            fld[comp] = f
            F.append(fld)
            Qb.append(-f * uxw if comp == 0 else (-f * utw if comp == 1 else f))
    F, Qb = np.stack(F), np.stack(Qb)

    # --interior: every characteristic of a window on times 2 .. Nt - 3, where u_t is the central five-point difference
    starts = list(range(2, Nt - args.K - 2, args.every) if args.interior else range(0, Nt - args.K, args.every))
    reg = slice(shrink, W - shrink)
    u0 = torch.tensor(np.concatenate([uw[:, :, j] for j in starts]))[:, None]
    # ground-truth free: the operator's rollout error on the scored region, 1 and K steps,
    # for u and for its window derivative (what d_x's residual differentiates)
    with torch.no_grad():
        s_ = u0
        for kk in range(1, args.K + 1):
            s_ = stp(s_)
            if kk in (1, args.K):
                off = shrink - kk * stp.r
                pred = s_[:, 0, off:off + (W - 2 * shrink)].numpy()
                tgt = np.concatenate([uw[:, reg, j + kk] for j in starts])
                cur = np.concatenate([uw[:, reg, j] for j in starts])
                dxr = np.tile(dx, len(starts))
                e_u = np.linalg.norm(pred - tgt) / np.linalg.norm(tgt - cur)
                e_d = (np.linalg.norm(dx4(pred[:, :, None], dxr)[:, 2:-2] - dx4(tgt[:, :, None], dxr)[:, 2:-2])
                       / np.linalg.norm(dx4(tgt[:, :, None], dxr)[:, 2:-2] - dx4(cur[:, :, None], dxr)[:, 2:-2]))
                print(f"  rollout error relative to the data increment after {kk} step(s): u {e_u:.3f}, u_x {e_d:.3f}")
    # ground-truth free: response in the scored region to a unit impulse at the window's first point,
    # pushed through the operator's Jacobian (zero for an operator whose influence cone excludes it)
    imp = torch.zeros_like(u0)
    imp[:, 0, 0] = 1.0
    s_, t_ = u0, imp
    leak = 0.0
    for kk in range(1, args.K + 1):
        s_, t_ = jvp(stp, (s_,), (t_,))
        off = shrink - kk * stp.r
        leak = max(leak, float(t_[:, 0, off:off + (W - 2 * shrink)].pow(2).mean(1).sqrt().mean()))
    print(f"  edge leak: rms response in the scored region to a unit impulse at the window edge, "
          f"max over {args.K} steps: {leak:.1e}")
    A, B, Q0 = [], [], []
    for p in range(Qb.shape[0]):
        q0 = np.concatenate([Qb[p][:, :, j] for j in starts])
        s, t = u0, torch.tensor(q0)[:, None]
        a_, b_ = [], []
        for k in range(1, args.K + 1):
            s, t = jvp(stp, (s,), (t,))
            off = shrink - k * stp.r
            tk = t[:, 0, off:off + (W - 2 * shrink)].numpy()
            a_.append(tk - q0[:, reg])
            b_.append(np.concatenate([Qb[p][:, reg, j + k] - Qb[p][:, reg, j] for j in starts]))
        A.append(np.stack(a_, -1).ravel())
        B.append(np.stack(b_, -1).ravel())
        Q0.append(q0[:, reg].ravel())
    A, B, Q0 = np.stack(A, 1), np.stack(B, 1), np.stack(Q0, 1)
    Nr = (A - B).T @ (A - B)
    G0 = args.K * Q0.T @ Q0
    Dr = A.T @ A + B.T @ B + args.mu * np.trace(B.T @ B) / np.trace(G0) * G0
    lam, C = e34.solve(Nr, Dr)
    # ground-truth free: how far the characteristics move over the horizon, relative to their size
    rel_change = float(np.sqrt(np.trace(B.T @ B) / np.trace(G0)))

    # the class measures x from each window's centre; the ground truth must too
    names, gt = e31.algebra(args.pde, xw - xw[:, W // 2:W // 2 + 1], ts, uw, cfg.alpha_heat, d)
    cg, *_ = np.linalg.lstsq(F.reshape(F.shape[0], -1).T, np.stack([g.ravel() for g in gt]).T, rcond=None)
    rq = [float(c @ Nr @ c / (c @ Dr @ c)) for c in cg.T]
    k = len(names)
    sel = (slice(None), slice(None), reg, slice(tm, -tm))
    gtv = np.stack([g[sel].ravel() for g in gt])
    Fv = F[(slice(None),) + sel].reshape(F.shape[0], -1)
    Qe, _ = np.linalg.qr((C[:, :k].T @ Fv).T)
    cont = [np.linalg.norm(Qe.T @ g) / np.linalg.norm(g) for g in gtv]
    print(f"[{args.pde}, {args.op} operator r = {stp.r}] seed {args.seed}, data seed {dseed}, {N} trajectories, window "
          f"{args.periods} periods ({W} points), scored on the central {W - 2 * shrink}, degree {d}, "
          f"K = {args.K}, mu_rel = {args.mu}, u_x {args.ux}, noise {args.noise:g}, thin {args.thin}")
    print(f"  relative change of the basis characteristics, rms over k = 1..K steps: {rel_change:.1e}")
    print("  lowest eigenvalues: " + " ".join(f"{v:.1e}" for v in lam[:k + 3]))
    print("  algebra losses: " + "  ".join(f"{n} {v:.1e}" for n, v in zip(names, rq)) + f"   (lambda{k + 1} = {lam[k]:.1e})")
    print(f"  algebra inside the {k} lowest: " + "  ".join(f"{n} {c:.2f}" for n, c in zip(names, cont)))
    if args.save:
        # eigen-generator j = sum_i C[i, j] * monomial_i in component i // len(mons), monomials in
        # X = (x - x_offset) / Lw, T = t / T_max, U = u / sigma, x the torus coordinate of xs
        np.savez(args.save, pde=args.pde, method=f"FINS-{args.op}", seed=args.seed, data_seed=dseed,
                 C=C[:, :k + 3], lam=lam[:k + 6], mons=np.array(mons), degree=d, Lw=Lw, T_max=T,
                 sigma=uw.std(), x_offset=dx * ((W // 2) % Nx), u=u_clean, xs=xs, ts=ts, names=np.array(names),
                 losses=np.array(rq), containment=np.array(cont), rel_change=rel_change,
                 band=np.array([su, sut]), K=args.K, k=k, noise=args.noise, thin=args.thin, interior=args.interior)
        print(f"  saved {args.save}")
    if args.m:
        rms = np.sqrt((F.reshape(F.shape[0], -1) ** 2).mean(1))
        Rot = e32.varimax(np.linalg.qr(C[:, :args.m] * rms[:, None])[0])
        scale = (Lw, T, uw.std())
        for i in range(args.m):
            w = Rot[:, i]
            c = w / rms
            big = np.abs(w).max()
            terms = []
            for j in np.argsort(-np.abs(w)):
                if abs(w[j]) < 0.05 * big:
                    break
                comp, (a, b, cc) = j // len(mons), mons[j % len(mons)]
                terms.append((comp, c[j] / (scale[0] ** a * scale[1] ** b * scale[2] ** cc),
                              e32.monomial_label(a, b, cc)))
            lead = terms[0][1]
            loss = float(c @ Nr @ c / (c @ Dr @ c))
            rest = float(np.sqrt(sum(w[q] ** 2 for q in range(len(w)) if abs(w[q]) < 0.05 * big)) / np.linalg.norm(w))
            form = ";  ".join(f"{('xi', 'eta', 'phi')[q]} = " + " ".join(f"{r / lead:+.3g} {lab}" for cmp, r, lab in terms if cmp == q)
                              for q in range(3) if any(t_[0] == q for t_ in terms))
            print(f"  v{i + 1}  loss {loss:.1e}  {len(terms)} terms (rest {rest:.2f}):  {form}   (x from the window centre)")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /content/fins/fins_repo/experiments/exp48_learned.py
"""Experiment 48, learned arm -- FINS through learned operators under noise and thinning (README_EXPERIMENTS.md).

exp48_noise_thinning.py compared the criteria with perfect models and ran LieNLSD as published (network F fitted to
the corrupted training data). This arm gives FINS's row with learned operators on the same corrupted data, so that
the two methods are compared with the models each of them learns: FINS a solution operator, LieNLSD a network for
u_t. Per condition, seeds 42-44, evaluation data 1000 + s, exp48's windows (snapshots 2 .. Nt - 3, the same physical
horizon at every thinning) and u_t (five-point differences of the corrupted snapshots; for KdV in place of the
protocol's solver stencil):
  clean     the protocol's operators, loaded (S_theta_heat_s{s}_g64x128_cno_smooth_L1.pt,
            S_theta_kdv_s{s}_mixed_stepper.pt in FINS_RESULTS_DIR), no training;
  clean-rt  operators retrained on the clean training data with the protocol's settings: the control for the
            corrupted conditions when those are trained on another device than the protocol's operators;
  others    operators retrained on the equally corrupted training data (protocol settings, 400 epochs; the heat
            CNO is cropped by 6 x thin points per training step, unrolled over at most 16 original steps (4 kept
            steps, 2 at s = 8, where 4 would leave no target in the 256-point window) and scored on the protocol's
            region, 96 points from each window edge).
The commands are those of the Colab cells "exp48". Generators are scored on the clean evaluation trajectories (exp42,
--t-margin 8 // s); the perfect-model rows come from exp48's summary.json.

Run:  python exp48_learned.py run --pdes kdv --conds n1e-4,t4 [--seeds 42,43,44] [--threads 1]
      python exp48_learned.py summary
Workers may run side by side: each claims a job by creating <output>.claim with the boot id and its pid; a claim
from before a reboot or from a dead process is taken over (--reset-claims removes every claim, with no worker
running). Checkpoints go to FINS_RESULTS_DIR with the condition in their tag, and training writes a partial state
every 10 epochs, so a stopped run resumes where it was.
"""

import argparse
import glob
import json
import os
import shutil
import subprocess
import sys
import time

import numpy as np

HERE = os.path.dirname(os.path.abspath(__file__))
CONDS = {"clean": (0.0, 1), "clean-rt": (0.0, 1), "n1e-4": (1e-4, 1), "n3e-4": (3e-4, 1), "n1e-3": (1e-3, 1),
         "n1e-2": (1e-2, 1), "t2": (0.0, 2), "t4": (0.0, 4), "t8": (0.0, 8)}
PERFECT = ["FINS, exact operator", "LieNLSD_RHS@eval", "LieNLSD_RHS_spec@eval", "LieNLSD_F", "LieNLSD_F@eval",
           "null: Gaussian coefficients"]


def learned_jobs(conds, pdes, seeds, runs):
    out = []
    for c in conds:
        sigma, s = CONDS[c]
        d = os.path.join(runs, c)
        corr = ["--K", str(16 // s), "--every", str(16 // s), "--noise", f"{sigma:g}", "--thin", str(s)]
        for seed in seeds:
            ds = ["--seed", str(seed), "--data-seed", str(1000 + seed)]
            for pde in pdes:
                f = os.path.join(d, f"fins_{pde}_s{seed}.npz")
                if pde == "heat":
                    base = ["exp40_window.py", "--pde", "heat", "--op", "cno", "--smooth", "--only-L", "1", "--n", "10"]
                    if c == "clean":
                        cmd = [*base, "--tag", "_smooth_L1", *ds, "--interior", "--save", f]
                    else:
                        # unroll x thin <= 16 original steps: at s = 8 an unroll of 4 would crop 4 x 48 points per
                        # side of the 256-point window, leaving no target (the protocol's 4 is kept for s <= 4)
                        cmd = [*base, "--tag", f"_smooth_L1_{c}", "--train-if-missing", *ds, "--score-margin", "96",
                               "--train-margin", str(6 * s), "--unroll", str(min(4, 16 // s)), *corr, "--interior",
                               "--save", f]
                else:
                    base = ["exp39_localised.py", "--pde", "kdv", "--op", "stepper", "--ic", "periodic", "--train-ic",
                            "mixed", "--n", "10"]
                    if c == "clean":
                        cmd = [*base, *ds, "--ut", "fd", "--interior", "--save", f]
                    else:
                        cmd = [*base, "--tag", f"_{c}", "--train-if-missing", *ds, "--ut", "fd", *corr, "--interior",
                               "--save", f]
                out.append((f, cmd))
    return out


def done(f):
    return os.path.exists(f) and not os.path.exists(f + ".claim")


def boot_id():
    try:
        return open("/proc/sys/kernel/random/boot_id").read().strip()
    except OSError:
        return "unknown"


def stale(claim):
    """A claim written before the last reboot, or by a process that no longer runs, holds nothing."""
    try:
        bid, pid = open(claim).read().split()
    except (OSError, ValueError):
        return False                              # being written, or an old empty claim: leave it
    if bid != boot_id():
        return True
    try:
        os.kill(int(pid), 0)
    except ProcessLookupError:
        return True
    except PermissionError:
        pass
    return False


def run(args):
    todo = learned_jobs(args.conds.split(","), args.pdes.split(","), [int(x) for x in args.seeds.split(",")],
                        args.runs)
    if args.reset_claims:
        for p in glob.glob(os.path.join(args.runs, "*", "*.claim")):
            os.remove(p)
    env = dict(os.environ, OMP_NUM_THREADS=str(args.threads), MKL_NUM_THREADS=str(args.threads),
               OPENBLAS_NUM_THREADS=str(args.threads))
    failed = set()
    while True:
        left = [j for j in todo if j[0] not in failed and not done(j[0])]
        if not left:
            break
        ran = False
        for f, cmd in left:
            os.makedirs(os.path.join(os.path.dirname(f), "logs"), exist_ok=True)
            claim = f + ".claim"
            if done(f):
                continue
            if os.path.exists(claim) and stale(claim):
                os.remove(claim)
            try:
                fd = os.open(claim, os.O_CREAT | os.O_EXCL | os.O_WRONLY)
            except FileExistsError:
                continue                          # another worker holds this job
            os.write(fd, f"{boot_id()} {os.getpid()}".encode())
            os.close(fd)
            log = os.path.join(os.path.dirname(f), "logs", os.path.basename(f).replace(".npz", ".log"))
            t0 = time.time()
            try:
                with open(log, "a") as fh:
                    fh.write("$ " + " ".join(cmd) + "\n")
                    fh.flush()
                    rc = subprocess.call([sys.executable, "-u", *cmd], cwd=HERE, stdout=fh, stderr=subprocess.STDOUT,
                                         env=env)
            finally:
                os.remove(claim)
            ok = rc == 0 and done(f)
            if not ok:
                failed.add(f)
            ran = True
            print(f"{'ok' if ok else 'FAILED'} {time.time() - t0:6.0f} s  {f}", flush=True)
            break                                 # rescan: claims may have been released meanwhile
        if not ran:
            time.sleep(20)


def summary(args):
    perfect = json.load(open(os.path.join(args.out, "summary.json")))
    seeds = [int(x) for x in args.seeds.split(",")]
    rows, seen, dt = {}, {}, {}
    for c, (sigma, s) in CONDS.items():
        d = os.path.join(args.runs, c)
        dst = os.path.join(args.out, c, "learned")
        js = os.path.join(dst, "scores.json")
        npz = glob.glob(os.path.join(d, "fins_*.npz"))
        if npz:
            pdes = sorted({os.path.basename(p).split("_")[1] for p in npz})
            os.makedirs(os.path.join(dst, "logs"), exist_ok=True)
            with open(os.path.join(dst, "tables.md"), "w") as fh:
                subprocess.check_call([sys.executable, "-u", "exp42_tables.py", "--dir", d, "--pdes", ",".join(pdes),
                                       "--seeds", args.seeds, "--t-margin", str(max(1, 8 // s)), "--json", js],
                                      cwd=HERE, stdout=fh, stderr=subprocess.STDOUT)
            for p in glob.glob(os.path.join(d, "logs", "*.log")):
                shutil.copy(p, os.path.join(dst, "logs"))
        if not os.path.exists(js):
            continue
        for pde, tab in json.load(open(js)).items():
            if "FINS" not in tab:
                continue
            # exp42 scores the seeds present in seed order; keep their labels for per-seed comparisons
            seen[(pde, c)] = [x for x in seeds if os.path.exists(os.path.join(d, f"fins_{pde}_s{x}.npz"))]
            for cv, v in tab["FINS"].items():
                rows.setdefault(pde, {}).setdefault(f"FINS, learned operator [{cv}]", {})[c] = v["missing"]
            names = tab["_names"]
            dt[(pde, c)] = [cs[names.index("∂_t")] for cs in tab["FINS"]["finite"]["contain"]]
    for pde in rows:
        for lab, r in perfect.get(pde, {}).items():
            if any(lab.startswith(p + " [") or (p.startswith("null") and lab.startswith(p)) for p in PERFECT):
                rows[pde][lab] = {c: v for c, v in r.items() if c != "protocol"}
    with open(os.path.join(args.out, "summary_learned.json"), "w") as fh:
        json.dump({"rows": rows, "seeds": {f"{p} {c}": v for (p, c), v in seen.items()},
                   "dt_contain": {f"{p} {c}": v for (p, c), v in dt.items()}}, fh, indent=1)
    conds = [c for c in CONDS if any(c in r for t in rows.values() for r in t.values())]
    with open(os.path.join(args.out, "summary_learned.md"), "w") as fh:
        for pde, tab in rows.items():
            fh.write(f"## {pde}: missing dimensions, m = k, mean +/- sd over seeds (n); per seed in the JSON; clean-rt: "
                     f"operators retrained on clean data (the perfect-model rows are those of clean)\n\n")
            fh.write("| row | " + " | ".join(conds) + " |\n" + "|---" * (len(conds) + 1) + "|\n")
            for lab, r in tab.items():
                cells = [f"{np.mean(r[c]):.2f} +/- {np.std(r[c]):.2f} ({len(r[c])})" if c in r else "" for c in conds]
                fh.write(f"| {lab} | " + " | ".join(cells) + " |\n")
            fh.write("\n")
        fh.write("## Registered predictions (README, learned arm), evaluated on the seeds present\n\n")
        for line in verdicts(rows, seen, dt, seeds):
            fh.write(line + "\n")
    print(open(os.path.join(args.out, "summary_learned.md")).read())


def verdicts(rows, seen, dt, seeds):
    """The registered predictions, literally; 'incomplete' while a condition lacks seeds."""
    def get(pde, lab, c):
        r = rows.get(pde, {}).get(lab, {})
        return r.get(c)

    def fins(pde, c):
        v = get(pde, "FINS, learned operator [finite]", c)
        return (np.array(v), seen[(pde, c)]) if v is not None else (None, [])

    def perfect(pde, lab, c, on):
        v = get(pde, lab, c)
        return None if v is None else np.array([v[seeds.index(x)] for x in on])

    def full(pde, c):
        return len(fins(pde, c)[1]) == len(seeds)

    fmt = lambda a: " / ".join(f"{x:.2f}" for x in a)
    out = []
    v, on = fins("heat", "clean")
    if v is not None:
        out.append(f"(C1) heat clean, protocol CNOs: {v.mean():.2f} (per seed {fmt(v)}) against 0.14 +/- 0.10 -> "
                   f"{'met' if abs(v.mean() - 0.14) <= 0.10 else 'NOT MET'}" + ("" if full("heat", "clean") else
                                                                                    " (incomplete)"))
    v, on = fins("kdv", "clean")
    if v is not None:
        out.append(f"(C1) KdV clean, protocol steppers: {v.mean():.2f} (per seed {fmt(v)}) against <= 0.05 -> "
                   f"{'met' if v.mean() <= 0.05 else 'NOT MET'}" + ("" if full("kdv", "clean") else " (incomplete)"))
    for pde, bound in (("kdv", None), ("heat", 0.15)):
        v, on = fins(pde, "clean-rt")
        if v is None:
            continue
        ok = (v <= 0.05).all() if pde == "kdv" else abs(v.mean() - 0.14) <= bound
        out.append(f"(LC) {pde} clean-rt: {v.mean():.2f} (per seed {fmt(v)}) -> {'met' if ok else 'NOT MET'}"
                   + ("" if full(pde, "clean-rt") else " (incomplete)"))
    for pde, tag in (("kdv", "LT1"), ("heat", "LT2")):
        parts, oks, comp = [], [], True
        ref, _ = fins(pde, "clean-rt")
        for c in ("t2", "t4", "t8"):
            v, on = fins(pde, c)
            if v is None:
                comp = False
                continue
            comp &= full(pde, c)
            F = perfect(pde, "LieNLSD_F [finite]", c, on)
            if pde == "kdv":
                oks += [v.mean() <= 0.10, bool((v < F).all())]
            else:
                oks += [ref is not None and abs(v.mean() - ref.mean()) <= 0.15, v.mean() < F.mean()]
            parts.append(f"{c}: {v.mean():.2f} ({fmt(v)}) vs F {F.mean():.2f} ({fmt(F)})")
        if parts:
            out.append(f"({tag}) {pde} thinning: " + "; ".join(parts) + (f"; clean-rt {ref.mean():.2f}"
                       if ref is not None and pde == "heat" else "") + f" -> {'met' if all(oks) else 'NOT MET'}"
                       + ("" if comp else " (incomplete)"))
    for pde, tag in (("heat", "LN1"), ("kdv", "LN2")):
        parts, oks, comp = [], [], True
        for c in ("n1e-4", "n3e-4", "n1e-3"):
            v, on = fins(pde, c)
            if v is None:
                comp = False
                continue
            comp &= full(pde, c)
            E = perfect(pde, "FINS, exact operator [finite]", c, on)
            oks.append(v.mean() >= E.mean() - 0.05 if pde == "heat" else abs(v.mean() - E.mean()) <= 0.15)
            parts.append(f"{c}: {v.mean():.2f} ({fmt(v)}) vs exact {E.mean():.2f}")
        if parts:
            out.append(f"({tag}) {pde} noise: " + "; ".join(parts) + f" -> {'met' if all(oks) else 'NOT MET'}"
                       + ("" if comp else " (incomplete)"))
    parts, oks, comp = [], [], True
    for pde, c, rule in (("kdv", "n1e-4", "fewer, per seed"), ("kdv", "n3e-4", "fewer, per seed"),
                         ("kdv", "n1e-2", "more, mean"), ("heat", "n1e-2", "more, mean")):
        v, on = fins(pde, c)
        if v is None:
            comp = False
            continue
        comp &= full(pde, c)
        F = perfect(pde, "LieNLSD_F [finite]", c, on)
        oks.append(bool((v < F).all()) if rule.startswith("fewer") else v.mean() > F.mean())
        parts.append(f"{pde} {c} ({rule}): {fmt(v)} vs F {fmt(F)}")
    if parts:
        out.append("(LN3) " + "; ".join(parts) + f" -> {'met' if all(oks) else 'NOT MET'}"
                   + ("" if comp else " (incomplete)"))
    parts, oks, comp = [], [], True
    for pde in ("heat", "kdv"):
        for c in ("n1e-3", "n1e-2"):
            if (pde, c) not in dt:
                comp = False
                continue
            comp &= full(pde, c)
            d_ = np.array(dt[(pde, c)])
            oks.append((d_ < 0.9).sum() >= 2)
            parts.append(f"{pde} {c}: d_t containment {fmt(d_)}")
    if parts:
        out.append("(LN4) " + "; ".join(parts) + f" -> {'met' if all(oks) else 'NOT MET'}"
                   + ("" if comp else " (incomplete)"))
    return out


def latex(args):
    """Rows of the paper's noise tables from summary_learned.json: means (Table 5) and means +/- sd (Table 11)."""
    rows = json.load(open(os.path.join(args.out, "summary_learned.json")))["rows"]
    order = ["clean", "n1e-4", "n3e-4", "n1e-3", "n1e-2", "t2", "t4", "t8"]
    for pde, cv, title in (("heat", "finite", "heat (6)"), ("heat", "full", "heat, class (10)"),
                           ("kdv", "finite", "KdV (4)")):
        r = rows.get(pde, {}).get(f"FINS, learned operator [{cv}]", {})
        mean = [f"{np.mean(r[c]):.2f}" if c in r else "--" for c in order]
        sd = [f"{np.mean(r[c]):.2f}\\,$\\pm$\\,{np.std(r[c]):.2f}" if c in r else "--" for c in order]
        print(f"% {title}; seeds per condition: " + ", ".join(f"{c} {len(r[c])}" for c in order if c in r))
        print("FINS, learned operator & " + " & ".join(mean) + " \\\\")
        print("FINS, learned operator & " + " & ".join(sd) + " \\\\")
        if "clean-rt" in r:
            print(f"% clean-rt (retrained on clean data): {np.mean(r['clean-rt']):.2f} +/- {np.std(r['clean-rt']):.2f}")


def main():
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("what", choices=["run", "summary", "latex"])
    ap.add_argument("--conds", default="clean,n1e-4,t4")
    ap.add_argument("--pdes", default="heat,kdv")
    ap.add_argument("--seeds", default="42,43,44")
    ap.add_argument("--runs", default=os.path.join(os.environ.get("FINS_RESULTS_DIR", "results"),
                                                   "noise_thin_learned_runs"), help="npz files and logs")
    ap.add_argument("--out", default=os.path.join(os.environ.get("FINS_RESULTS_DIR", "results"), "noise_thin"),
                    help="exp48's summary folder: learned tables go to <cond>/learned/, the merged table to "
                         "summary_learned.md")
    ap.add_argument("--threads", type=int, default=1)
    ap.add_argument("--reset-claims", action="store_true")
    args = ap.parse_args()
    args.runs, args.out = (p if os.path.isabs(p) else os.path.join(HERE, p) for p in (args.runs, args.out))
    {"run": run, "summary": summary, "latex": latex}[args.what](args)


if __name__ == "__main__":
    main()

In [ ]:
# 3. run: 24 CNO trainings (8 conditions x 3 seeds) and FINS through each; output streams one line per finished run
LR = os.path.join(os.environ['FINS_RESULTS_DIR'], 'noise_thin_learned_runs')
cmd = [sys.executable, '-u', 'exp48_learned.py', 'run', '--pdes', 'heat',
       '--conds', 'clean-rt,n1e-4,t4,t8,t2,n3e-4,n1e-3,n1e-2', '--threads', '2', '--runs', LR]
t0 = time.time()
with subprocess.Popen(cmd, cwd=EXPDIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1) as p:
    for line in p.stdout:
        print(line, end='', flush=True)
print(f'exit {p.returncode} after {time.time() - t0:.0f} s')

In [ ]:
# 4. progress (safe while cell 3 runs in another tab, or after it stopped): finished runs and the tail of each log
LR = os.path.join(os.environ['FINS_RESULTS_DIR'], 'noise_thin_learned_runs')
for d in sorted(glob.glob(os.path.join(LR, '*'))):
    print(os.path.basename(d), sorted(os.path.basename(f) for f in glob.glob(os.path.join(d, 'fins_*.npz'))))
    for log in sorted(glob.glob(os.path.join(d, 'logs', '*.log'))):
        lines = [l for l in open(log).read().splitlines() if l.strip()]
        print('   ', os.path.basename(log), '|', lines[-1][:150] if lines else '')

In [ ]:
# 5. zip the outputs (npz + logs; checkpoints stay on Drive) and download
import shutil
from google.colab import files
LR = os.path.join(os.environ['FINS_RESULTS_DIR'], 'noise_thin_learned_runs')
arc = shutil.make_archive('/content/fins_noise_thin_learned_heat', 'zip', LR)
shutil.copy(arc, '/content/drive/MyDrive/fins_noise_thin_learned_heat.zip')
print(arc, os.path.getsize(arc), 'bytes; a copy is on Drive')
files.download(arc)